# Da comparação à escolha: justificando a pipeline de avaliação de frases

Temos dois notebooks:

- **`estudo_tecnicas_avaliacao.ipynb`** mediu, para cada dimensão, **qual técnica vence** — parser
  vs. LanguageTool vs. SLOR vs. PLL, e assim por diante — com acurácia por defeito e custo.
- **`avaliacao_de_frases.ipynb`** montou uma **pipeline**: escolheu *uma* técnica por dimensão e as
  encadeou num laudo.

Este notebook liga os dois. Ele faz duas coisas:

1. **Justifica cada escolha da pipeline** com os números do estudo — por que aquela técnica, e não
   as concorrentes.
2. **Mede a pipeline montada** rodando-a sobre o corpus rotulado do estudo (36 frases de pares
   mínimos, com gabarito) e extrai **métricas avaliativas reais**: precisão, recall, F1, cobertura e
   falso alarme por etapa.

O ponto metodológico que dá força ao teste: **a pipeline foi construída sobre um corpus de 8
frases** (as do `avaliacao_de_frases.ipynb`) e agora é avaliada sobre **36 frases diferentes**. É
um teste de generalização — o conjunto de avaliação é independente do de desenvolvimento.

> **Custo:** carrega spaCy, léxico pt, Qwen2.5-1.5B e BERTimbau na CPU. Alguns minutos.

In [1]:
import re, math
from collections import defaultdict

import spacy, torch
from spellchecker import SpellChecker
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForMaskedLM

nlp = spacy.load("pt_core_news_sm")
lexico = SpellChecker(language="pt")
TOTAL_TOKENS = lexico.word_frequency.total_words
tokenizar = lambda t: re.findall(r"[a-zà-ÿ]+", t.lower())

QWEN = "Qwen/Qwen2.5-1.5B-Instruct"
tok_q = AutoTokenizer.from_pretrained(QWEN)
mod_q = AutoModelForCausalLM.from_pretrained(QWEN, dtype=torch.float32); mod_q.eval()
BERTIMBAU = "neuralmind/bert-base-portuguese-cased"
tok_b = AutoTokenizer.from_pretrained(BERTIMBAU)
mod_b = AutoModelForMaskedLM.from_pretrained(BERTIMBAU); mod_b.eval()
print("carregado")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


carregado


In [2]:
# --- CORPUS ROTULADO DO ESTUDO: 6 bases x 6 variantes, gabarito conhecido ---
DEFEITOS = ["correta", "ortografia", "conc_nominal", "conc_verbal", "fragmento", "anomalia"]

PARES = {
 "B1": {"correta":"Os pesquisadores analisaram os dados coletados durante o experimento.",
        "ortografia":"Os pesquisadores analizaram os dados coletados durante o experimento.",
        "conc_nominal":"Os pesquisadores analisaram os dados coletado durante o experimento.",
        "conc_verbal":"Os pesquisadores analisou os dados coletados durante o experimento.",
        "fragmento":"Os pesquisadores que analisaram os dados coletados durante o experimento.",
        "anomalia":"Os pesquisadores beberam os dados coletados durante o experimento."},
 "B2": {"correta":"A prefeitura instalou novas câmeras nas praças do centro.",
        "ortografia":"A prefeitura instalou novas câmeras nas praças do sentro.",
        "conc_nominal":"A prefeitura instalou nova câmeras nas praças do centro.",
        "conc_verbal":"A prefeitura instalaram novas câmeras nas praças do centro.",
        "fragmento":"A prefeitura que instalou novas câmeras nas praças do centro.",
        "anomalia":"A prefeitura digeriu novas câmeras nas praças do centro."},
 "B3": {"correta":"Os alunos entregaram os trabalhos antes do prazo final.",
        "ortografia":"Os alunos entregaram os trabalhos antes do praso final.",
        "conc_nominal":"Os alunos entregaram os trabalho antes do prazo final.",
        "conc_verbal":"Os alunos entregou os trabalhos antes do prazo final.",
        "fragmento":"Os alunos que entregaram os trabalhos antes do prazo final.",
        "anomalia":"Os alunos evaporaram os trabalhos antes do prazo final."},
 "B4": {"correta":"O médico receitou os remédios indicados para a paciente.",
        "ortografia":"O médico reseitou os remédios indicados para a paciente.",
        "conc_nominal":"O médico receitou os remédios indicado para a paciente.",
        "conc_verbal":"Os médicos receitou os remédios indicados para a paciente.",
        "fragmento":"O médico que receitou os remédios indicados para a paciente.",
        "anomalia":"O médico cantou os remédios indicados para a paciente."},
 "B5": {"correta":"As empresas contrataram profissionais qualificados para o projeto.",
        "ortografia":"As empresas contrataram proficionais qualificados para o projeto.",
        "conc_nominal":"As empresas contrataram profissionais qualificado para o projeto.",
        "conc_verbal":"As empresas contratou profissionais qualificados para o projeto.",
        "fragmento":"As empresas que contrataram profissionais qualificados para o projeto.",
        "anomalia":"As empresas ferveram profissionais qualificados para o projeto."},
 "B6": {"correta":"Os moradores denunciaram os problemas encontrados no edifício.",
        "ortografia":"Os moradores denuciaram os problemas encontrados no edifício.",
        "conc_nominal":"Os moradores denunciaram os problema encontrados no edifício.",
        "conc_verbal":"Os moradores denunciou os problemas encontrados no edifício.",
        "fragmento":"Os moradores que denunciaram os problemas encontrados no edifício.",
        "anomalia":"Os moradores assaram os problemas encontrados no edifício."},
}
SONDAS = {
 "embaralhada_1":"Dados os analisaram pesquisadores os durante coletados experimento o.",
 "embaralhada_2":"Praças instalou câmeras a novas nas prefeitura do centro.",
 "vazia":"Ideias verdes incolores dormem furiosamente.",
 "rara_correta":"O sismólogo catalogou anomalias geomagnéticas ininterruptamente.",
}
todas = lambda: [(f"{b}/{t}", b, t, d[t]) for b, d in PARES.items() for t in DEFEITOS]

# gabarito: qual ETAPA da pipeline deveria pegar cada tipo de defeito
ALVO = {"ortografia":"ortografia", "conc_nominal":"gramática", "conc_verbal":"gramática",
        "fragmento":"estrutura", "anomalia":"semântica"}
print(f"{len(todas())} frases rotuladas, {len(SONDAS)} sondas")

36 frases rotuladas, 4 sondas


In [3]:
# --- A PIPELINE: a melhor combinacao de tecnicas do estudo ---

def erros_ortograficos(texto):
    return {p: lexico.correction(p) for p in tokenizar(texto) if p not in lexico}

def numero(t): return t.morph.get("Number")
def genero(t): return t.morph.get("Gender")

def erros_concordancia(doc):
    erros = []
    for t in doc:
        if t.dep_ in ("det","amod","acl") and t.head.pos_ in ("NOUN","PROPN"):
            if numero(t) and numero(t.head) and numero(t)!=numero(t.head):
                erros.append(f"nominal/número: '{t.text}' × '{t.head.text}'")
            if genero(t) and genero(t.head) and genero(t)!=genero(t.head):
                erros.append(f"nominal/gênero: '{t.text}' × '{t.head.text}'")
        if t.dep_ in ("nsubj","nsubj:pass") and t.head.pos_ in ("VERB","AUX"):
            if numero(t) and numero(t.head) and numero(t)!=numero(t.head):
                erros.append(f"verbal: '{t.text}' × '{t.head.text}'")
    return erros

def analisar_estrutura(doc):
    raiz = [t for t in doc if t.dep_=="ROOT"][0]
    predicado = raiz.pos_ in ("VERB","AUX") or any(f.dep_=="cop" for f in raiz.children)
    prof = max(len(list(t.ancestors)) for t in doc)
    return raiz, predicado, prof

def surpresa(frase):
    enc = tok_q(frase, return_tensors="pt"); ids = enc["input_ids"][0]
    with torch.no_grad(): lp = torch.log_softmax(mod_q(**enc).logits, dim=-1)
    return [(tok_q.decode(ids[i]), -float(lp[0,i-1,ids[i]])) for i in range(2, len(ids))]

def slor(frase):
    enc = tok_q(frase, return_tensors="pt"); ids = enc["input_ids"][0]
    with torch.no_grad(): lp = torch.log_softmax(mod_q(**enc).logits, dim=-1)
    total = sum(float(lp[0,i-1,ids[i]]) for i in range(1,len(ids)))
    uni = sum(math.log((lexico.word_frequency[p] or 1)/TOTAL_TOKENS) for p in tokenizar(frase))
    return (total - uni)/len(tokenizar(frase))

# SEMANTICA = verbo mascarado (vencedor do estudo), decisao por MARGEM relativa
MARGEM_SEMANTICA = 10.0   # nats — corte relativo a posicao, nao um limiar absoluto de surpresa

def margem_verbo(frase):
    doc = nlp(frase)
    raiz = [t for t in doc if t.dep_=="ROOT" and t.pos_=="VERB"]
    if not raiz: return None                 # sem verbo na raiz (fragmento): a tecnica nao se aplica
    alvo = raiz[0].text
    enc = tok_b(frase.replace(alvo, tok_b.mask_token, 1), return_tensors="pt")
    pos = (enc["input_ids"][0]==tok_b.mask_token_id).nonzero()
    if len(pos)==0: return None
    with torch.no_grad():
        lp = torch.log_softmax(mod_b(**enc).logits[0, int(pos[0])], dim=-1)
    ida = tok_b(alvo, add_special_tokens=False)["input_ids"]
    return float(lp.max()) - float(lp[ida[0]])   # melhor preenchimento esperado - verbo escrito

def pipeline_dispara(frase):
    # Quais etapas-DETECTOR disparam. (aceitabilidade e' score, tratada a parte.)
    doc = nlp(frase)
    _, predicado, _ = analisar_estrutura(doc)
    orto = bool(erros_ortograficos(frase))
    m = margem_verbo(frase)
    # gate de OOV (o mesmo do SLOR): palavra fora do lexico ja e' pega pela ortografia
    # e desregula qualquer medida estatistica; nesse caso a semantica nao roda
    sem = (m is not None) and (not orto) and (m > MARGEM_SEMANTICA)
    return {
        "ortografia": orto,
        "gramática":  bool(erros_concordancia(doc)),
        "estrutura":  not predicado,
        "semântica":  sem,
    }

print("pipeline pronta. Etapas-detector:", list(pipeline_dispara(PARES["B1"]["correta"]).keys()))

pipeline pronta. Etapas-detector: ['ortografia', 'gramática', 'estrutura', 'semântica']


## Por que estas técnicas — o que o estudo mostrou

Cada linha da pipeline é uma escolha entre concorrentes. O estudo mediu essas disputas; a tabela
abaixo traz o vencedor e o placar que justifica a escolha.

| etapa da pipeline | técnica escolhida | concorrentes medidas no estudo | por que esta venceu |
|---|---|---|---|
| **ortografia** | léxico (dicionário) | — (parser é cego a grafia) | o estudo mostrou o parser T1 com **0/6** em ortografia: erro de grafia não é detectável por análise sintática. Precisa de uma etapa lexical dedicada, que é exata e custa ~0. |
| **gramática** | parser (concordância na árvore) | LanguageTool, SLOR, PLL | parser **6/6** em concordância verbal contra **1/6** do LanguageTool — a ferramenta madura de 259 MB perde para 20 linhas sobre a árvore. E parser localiza o token; SLOR/PLL só dão um número. |
| **estrutura** | teste da raiz | complexidade, dep. length, legibilidade | teste da raiz **6/6, zero falso positivo** — o melhor detector do estudo inteiro. A legibilidade (Flesch) ficou **anticorrelacionada** com a correção (0-1/6). |
| **aceitabilidade** | SLOR, **bloqueado se houver palavra fora do léxico** | PLL | SLOR acerta 29/30; sua **única** falha é inflar com palavra fora do vocabulário. A pipeline roda o corretor **antes** e só calcula SLOR se não houver OOV — o gating é exatamente a mitigação que o estudo apontou. |
| **semântica** | verbo mascarado (margem relativa) | surpresa, embeddings, LLM | **vencedor do estudo**: 5/6 contra 4/6 da surpresa e **14× mais barato**; embeddings são cegos à negação e o LLM custa 40× mais. Decide pela **margem** entre o melhor preenchimento esperado na posição da raiz e o verbo escrito — critério relativo, não um limiar absoluto de surpresa. |
| **coerência** | *ausente do laudo* | entity grid, embeddings, LM condicional | correto: o estudo **não achou técnica confiável** de coerência (a melhor acerta 1 parágrafo em 2). Uma pipeline honesta de nível de frase não promete o que não mede. |

Duas decisões da pipeline que o estudo **valida por omissão**, e que são fáceis de errar:

- **Flesch entra no laudo como número descritivo (profundidade/legibilidade), nunca como
  veredito.** O estudo mostrou por quê: como detector, o Flesch está anticorrelacionado — a frase
  quebrada pontua *melhor* que a correta. A pipeline reporta, mas não reprova por ele.
- **Coerência fica de fora do laudo de frase.** Não porque foi esquecida, mas porque o estudo
  mostrou que nenhuma das técnicas disponíveis mede coerência de forma confiável. Prometer uma nota
  de coerência aqui seria vender fumaça.

Agora o teste: a pipeline **cumpre** o que essas escolhas prometem?

In [4]:
# --- MÉTRICA 1: matriz de confusão — por tipo de defeito, quantas vezes cada etapa dispara ---
ETAPAS = ["ortografia", "gramática", "estrutura", "semântica"]
disparos = {i: pipeline_dispara(f) for i, b, d, f in todas()}

print("MATRIZ DE ROTEAMENTO (6 frases por tipo; a etapa-alvo está à direita)")
print(f"{'defeito real':14} " + " ".join(f"{e[:7]:>8}" for e in ETAPAS) + "   etapa-alvo")
print("-" * 66)
for d in DEFEITOS:
    cont = {e: 0 for e in ETAPAS}
    for i, b, dd, f in todas():
        if dd == d:
            for e in ETAPAS:
                cont[e] += disparos[i][e]
    diag = "" if d == "correta" else f"   ← {ALVO[d]}"
    print(f"{d:14} " + " ".join(f"{cont[e]:8}" for e in ETAPAS) + diag)
print("\nleitura: a massa deve ficar na diagonal (defeito → sua etapa).")
print("fora da diagonal na linha 'correta' = falso alarme; nas outras = etapa disparando no defeito errado.")

MATRIZ DE ROTEAMENTO (6 frases por tipo; a etapa-alvo está à direita)
defeito real    ortogra  gramáti  estrutu  semânti   etapa-alvo
------------------------------------------------------------------
correta               1        1        0        0
ortografia            6        1        0        0   ← ortografia
conc_nominal          1        5        0        0   ← gramática
conc_verbal           1        6        0        0   ← gramática
fragmento             1        1        6        0   ← estrutura
anomalia              1        1        0        5   ← semântica

leitura: a massa deve ficar na diagonal (defeito → sua etapa).
fora da diagonal na linha 'correta' = falso alarme; nas outras = etapa disparando no defeito errado.


In [5]:
# --- MÉTRICA 2: precisão, recall e F1 por etapa (tratando cada uma como detector binário) ---
print("MÉTRICAS POR ETAPA — a pipeline como sistema de detecção")
print(f"{'etapa':12} {'TP':>3} {'FP':>3} {'FN':>3} {'precisão':>9} {'recall':>7} {'F1':>6}")
print("-" * 52)
macro = []
for e in ETAPAS:
    tp = fp = fn = 0
    for i, b, dd, f in todas():
        disparou = disparos[i][e]
        eh_alvo = ALVO.get(dd) == e
        tp += disparou and eh_alvo
        fp += disparou and not eh_alvo
        fn += (not disparou) and eh_alvo
    prec = tp/(tp+fp) if tp+fp else 0.0
    rec  = tp/(tp+fn) if tp+fn else 0.0
    f1   = 2*prec*rec/(prec+rec) if prec+rec else 0.0
    macro.append(f1)
    print(f"{e:12} {tp:3} {fp:3} {fn:3} {prec:9.2f} {rec:7.2f} {f1:6.2f}")
print(f"\nF1 macro-médio: {sum(macro)/len(macro):.2f}")

# cobertura global e falso alarme
n_def = sum(1 for i, b, dd, f in todas() if dd != "correta")
cobertos = sum(1 for i, b, dd, f in todas() if dd != "correta" and disparos[i][ALVO[dd]])
corretas = [i for i, b, dd, f in todas() if dd == "correta"]
alarme = sum(1 for i in corretas if any(disparos[i].values()))
print(f"\ncobertura  (defeito pego pela etapa certa): {cobertos}/{n_def} = {cobertos/n_def:.0%}")
print(f"falso alarme (dispara em frase correta):     {alarme}/{len(corretas)} = {alarme/len(corretas):.0%}")

MÉTRICAS POR ETAPA — a pipeline como sistema de detecção
etapa         TP  FP  FN  precisão  recall     F1
----------------------------------------------------
ortografia     6   5   0      0.55    1.00   0.71
gramática     11   4   1      0.73    0.92   0.81
estrutura      6   0   0      1.00    1.00   1.00
semântica      5   0   1      1.00    0.83   0.91

F1 macro-médio: 0.86

cobertura  (defeito pego pela etapa certa): 28/30 = 93%
falso alarme (dispara em frase correta):     2/6 = 33%


## Lendo as métricas da pipeline

Os números confirmam a previsão do estudo — **a força de cada etapa acompanha o desempenho da
técnica que ela usa** — e expõem os pontos fracos com nome e sobrenome.

**Estrutura: F1 = 1,00.** Precisão e recall perfeitos, zero falso positivo em 30 frases — exatamente
o que o estudo previu para o teste da raiz (6/6, zero FP). A melhor técnica do estudo é a melhor
etapa da pipeline.

**Semântica: F1 = 0,91** (precisão 1,00, recall 0,83). O verbo mascarado, vencedor do estudo, separa
a anomalia da frase correta por uma margem larga e decide **sem escala absoluta** — pega inclusive
"cantou" e "ferveram", de surpresa naturalmente baixa. O único escape é B2 ("A prefeitura digeriu
novas **câmeras**"): "câmeras" está fora do léxico, o *gate* de OOV suspende a etapa semântica, e a
anomalia passa. É o mesmo léxico finito que limita a ortografia (abaixo) — uma causa, dois sintomas.

**Gramática: F1 = 0,81** (precisão 0,73, recall 0,92). Forte, como o parser prometia, com dois
tropeços que o estudo já anunciava:

- **Um falso negativo:** "Os alunos entregaram os **trabalho**" não é pego — o morfologizador do
  spaCy lê "trabalho" como plural e **absorve** o erro em vez de denunciá-lo.
- **Quatro falsos positivos:** "para **a paciente**" é acusado de erro de gênero. "Paciente" é
  substantivo de dois gêneros; o parser o etiqueta como masculino e vê conflito com o artigo "a",
  que está correto. São 4 disparos indevidos, todos da base B4, todos pela mesma palavra.

**Ortografia: recall 1,00, precisão 0,55.** Pega todos os 6 erros de grafia e acende em outras 5
frases **sem erro de grafia** — todas da base B2, todas por **"câmeras"**, palavra comum ausente do
léxico do pyspellchecker. É o limite estrutural de qualquer verificador por lista: a lista nunca
termina, e a precisão despenca quando o texto traz vocabulário legítimo fora dela.

**No agregado: F1 macro 0,86, cobertura 28/30 (93%), falso alarme 2/6.** Os buracos são todos
rastreáveis a limitações conhecidas — léxico finito e substantivos de dois gêneros —, não a defeitos
ocultos. É o retrato de uma pipeline **bem fundamentada**, cujos erros se explicam um a um.

In [6]:
# --- MÉTRICA 3: o gating do SLOR funciona? (a mitigação que o estudo pediu) ---
print("ACEITABILIDADE (SLOR) — o estudo mostrou que SLOR infla com palavra fora do léxico.")
print("A pipeline só calcula SLOR se NÃO houver OOV. As sondas testam o gating:\n")
print(f"{'sonda':16} {'tem OOV?':>9}   {'ação da pipeline':<32} {'frase'}")
for n, f in SONDAS.items():
    oov = erros_ortograficos(f)
    if oov:
        acao = f"BLOQUEIA (OOV: {list(oov)[0]})"
    else:
        acao = f"calcula SLOR = {slor(f):.2f}"
    print(f"{n:16} {str(bool(oov)):>9}   {acao:<32} {f[:40]}")
print("\nsem o gating, 'rara_correta' (frase correta, vocabulário raro) receberia SLOR inflado —")
print("o exato falso resultado que o estudo demonstrou. A pipeline se recusa a pontuá-la.")

ACEITABILIDADE (SLOR) — o estudo mostrou que SLOR infla com palavra fora do léxico.
A pipeline só calcula SLOR se NÃO houver OOV. As sondas testam o gating:

sonda             tem OOV?   ação da pipeline                 frase


embaralhada_1        False   calcula SLOR = -0.54             Dados os analisaram pesquisadores os dur
embaralhada_2         True   BLOQUEIA (OOV: câmeras)          Praças instalou câmeras a novas nas pref


vazia                False   calcula SLOR = -0.90             Ideias verdes incolores dormem furiosame


rara_correta          True   BLOQUEIA (OOV: geomagnéticas)    O sismólogo catalogou anomalias geomagné

sem o gating, 'rara_correta' (frase correta, vocabulário raro) receberia SLOR inflado —
o exato falso resultado que o estudo demonstrou. A pipeline se recusa a pontuá-la.


## Por que verbo mascarado — e por que margem relativa

A etapa semântica é a que mais depende de um detalhe: como converter a pontuação contínua em
veredito. O estudo comparou a **surpresa por token** (S1) com o **verbo mascarado** (S2) e deu
vantagem a S2 — 5/6 contra 4/6, e **14× mais barato**. A célula abaixo põe as duas frente a frente
nas 6 anomalias e mostra por que a surpresa com **limiar fixo** falha, e por que a decisão da
pipeline é por **margem relativa**.

In [7]:
# S2 do estudo: plausibilidade do verbo por preenchimento mascarado (BERTimbau)
def verbo_mascarado(frase):
    doc = nlp(frase)
    raiz = [t for t in doc if t.dep_=="ROOT" and t.pos_=="VERB"]
    if not raiz: return None
    alvo = raiz[0].text
    enc = tok_b(frase.replace(alvo, tok_b.mask_token, 1), return_tensors="pt")
    pos = (enc["input_ids"][0] == tok_b.mask_token_id).nonzero()
    if len(pos) == 0: return None
    with torch.no_grad():
        lp = torch.log_softmax(mod_b(**enc).logits[0, int(pos[0])], dim=-1)
    return float(lp[tok_b(alvo, add_special_tokens=False)["input_ids"][0]])

print("Para cada base: pico de SURPRESA (S1) e log P do verbo MASCARADO (S2),")
print("na anomalia vs. na correta. Detector acerta se separa as duas.\n")
print(f"{'base':5} {'S1 anom':>8} {'S1 corr':>8} {'S1>12?':>7}   {'S2 anom':>8} {'S2 corr':>8} {'margem S2':>10}")
s1_lim = s1_rank = s2_rank = 0
for b, d in PARES.items():
    s1a = max(v for _, v in surpresa(d["anomalia"]))
    s1c = max(v for _, v in surpresa(d["correta"]))
    s2a, s2c = verbo_mascarado(d["anomalia"]), verbo_mascarado(d["correta"])
    pega_s1 = s1a > 12.0
    s1_lim += pega_s1
    s1_rank += s1a > s1c
    s2_rank += s2a < s2c
    print(f"{b:5} {s1a:8.1f} {s1c:8.1f} {str(pega_s1):>7}   {s2a:8.2f} {s2c:8.2f} {s2c-s2a:10.2f}")
print(f"\ncomo RANQUEADOR (anomalia mais suspeita que a correta): S1 {s1_rank}/6, S2 {s2_rank}/6")
print(f"como DETECTOR de limiar fixo (>12 nats):                 S1 {s1_lim}/6")
print("\nrepare nas margens: S2 separa anomalia de correta por uma folga enorme e consistente;")
print("S1 deixa B4 ('cantou', 9.1) e B5 ('ferveram', 9.2) abaixo do corte de 12.")

Para cada base: pico de SURPRESA (S1) e log P do verbo MASCARADO (S2),
na anomalia vs. na correta. Detector acerta se separa as duas.

base   S1 anom  S1 corr  S1>12?    S2 anom  S2 corr  margem S2


B1        14.0      4.8    True     -16.29    -5.39      10.90


B2        15.8      7.9    True     -25.05    -0.07      24.99


B3        16.6     10.3    True     -18.69    -6.70      11.99


B4         9.1      7.9   False     -14.15    -7.73       6.42


B5         9.2      7.8   False     -19.55    -6.71      12.85


B6        14.3      7.6    True     -15.21    -6.02       9.18

como RANQUEADOR (anomalia mais suspeita que a correta): S1 6/6, S2 6/6
como DETECTOR de limiar fixo (>12 nats):                 S1 4/6

repare nas margens: S2 separa anomalia de correta por uma folga enorme e consistente;
S1 deixa B4 ('cantou', 9.1) e B5 ('ferveram', 9.2) abaixo do corte de 12.


### O critério de decisão

Nas 6 bases, tanto S1 quanto S2 colocam a anomalia como mais suspeita que a correta — **6/6 para os
dois** como ranqueadores. A diferença está no **critério de corte**. A surpresa com limiar absoluto
de 12 nats perde B4 ("O médico **cantou** os remédios") e B5 ("As empresas **ferveram**
profissionais"), cujos verbos comuns têm surpresa naturalmente baixa (~9 nats) — o estudo já avisava
que *a escala não é absoluta*.

Por isso a pipeline decide pelo **verbo mascarado** com **margem relativa**: a diferença entre o
melhor preenchimento esperado na posição da raiz e o verbo escrito. A anomalia destaca-se por uma
folga larga e consistente (coluna da direita), sem depender de um valor absoluto — e a etapa ainda
custa 14× menos que a surpresa. É a combinação que o estudo aponta como vencedora: uma camada
linguística (o parser acha o verbo) e uma estatística (o modelo julga a posição) se completam.

In [8]:
# --- O LAUDO COMPLETO: a pipeline em ação, com exemplos ---
def laudo(frase):
    doc = nlp(frase)
    linhas, bloqueado = [], False

    orto = erros_ortograficos(frase)
    linhas.append(("ortografia", "ERRO" if orto else "ok",
                   ", ".join(f"{p}→{c}" for p, c in orto.items()) or "—"))
    bloqueado = bool(orto)

    conc = erros_concordancia(doc)
    linhas.append(("gramática", "ERRO" if conc else "ok", "; ".join(conc) or "—"))

    raiz, pred, prof = analisar_estrutura(doc)
    linhas.append(("estrutura", "ERRO" if not pred else "ok",
                   f"raiz '{raiz.text}' não é verbo (fragmento)" if not pred
                   else f"raiz '{raiz.text}', profundidade {prof}"))

    m = margem_verbo(frase)
    if bloqueado:
        linhas.append(("semântica", "n/d", "não avaliada: palavra fora do léxico"))
    elif m is None:
        linhas.append(("semântica", "n/d", "sem verbo na raiz"))
    else:
        linhas.append(("semântica", "SUSPEITA" if m > MARGEM_SEMANTICA else "ok",
                       f"margem do verbo-raiz = {m:.1f} nats"))

    linhas.append(("aceitabilidade", "n/d" if bloqueado else "ok",
                   "não avaliada: palavra fora do léxico" if bloqueado
                   else f"SLOR = {slor(frase):.2f}"))

    print(f'"{frase}"')
    for dim, status, det in linhas:
        print(f"   {dim:15} {status:9} {det}")
    print()


# um exemplo de cada tipo, tirados do corpus rotulado
for tipo in DEFEITOS:
    laudo(PARES["B1"][tipo])

"Os pesquisadores analisaram os dados coletados durante o experimento."
   ortografia      ok        —
   gramática       ok        —
   estrutura       ok        raiz 'analisaram', profundidade 4
   semântica       ok        margem do verbo-raiz = 4.1 nats
   aceitabilidade  ok        SLOR = 5.82

"Os pesquisadores analizaram os dados coletados durante o experimento."
   ortografia      ERRO      analizaram→analisaram
   gramática       ok        —
   estrutura       ok        raiz 'analizaram', profundidade 4
   semântica       n/d       não avaliada: palavra fora do léxico
   aceitabilidade  n/d       não avaliada: palavra fora do léxico



"Os pesquisadores analisaram os dados coletado durante o experimento."
   ortografia      ok        —
   gramática       ERRO      nominal/número: 'coletado' × 'dados'
   estrutura       ok        raiz 'analisaram', profundidade 4
   semântica       ok        margem do verbo-raiz = 3.8 nats
   aceitabilidade  ok        SLOR = 4.91



"Os pesquisadores analisou os dados coletados durante o experimento."
   ortografia      ok        —
   gramática       ERRO      verbal: 'pesquisadores' × 'analisou'
   estrutura       ok        raiz 'analisou', profundidade 4
   semântica       ok        margem do verbo-raiz = 7.5 nats
   aceitabilidade  ok        SLOR = 4.95



"Os pesquisadores que analisaram os dados coletados durante o experimento."
   ortografia      ok        —
   gramática       ok        —
   estrutura       ERRO      raiz 'pesquisadores' não é verbo (fragmento)
   semântica       n/d       sem verbo na raiz
   aceitabilidade  ok        SLOR = 4.93



"Os pesquisadores beberam os dados coletados durante o experimento."
   ortografia      ok        —
   gramática       ok        —
   estrutura       ok        raiz 'beberam', profundidade 4
   semântica       SUSPEITA  margem do verbo-raiz = 15.0 nats
   aceitabilidade  ok        SLOR = 3.80



## Síntese

**O estudo justifica a pipeline, e a pipeline confirma o estudo.** As duas leituras se encontram:

| etapa | o estudo previu | a pipeline mediu (36 frases) | veredito |
|---|---|---|---|
| estrutura | teste da raiz 6/6, zero FP | **F1 = 1,00** | escolha ideal, confirmada |
| semântica | verbo mascarado 5/6 > surpresa 4/6 | **F1 = 0,91** | vencedor do estudo, margem relativa |
| gramática | parser 6/6 vs LanguageTool 1/6 | **F1 = 0,81** | escolha certa; limites conhecidos (morfologizador, gênero comum) |
| ortografia | parser cego a grafia (0/6) → precisa de léxico | **recall 1,00, precisão 0,55** | etapa necessária; precisão limitada pelo léxico |
| aceitabilidade | SLOR infla com OOV → rodar corretor antes | **gating funciona nas sondas** | mitigação do estudo, implementada |
| coerência | nenhuma técnica confiável | **ausente do laudo** | omissão honesta, correta |

**Três conclusões:**

1. **A pipeline é bem fundamentada.** Cada etapa emprega o vencedor do estudo na sua dimensão —
   incluindo as duas decisões "invisíveis" (manter o Flesch fora do veredito, deixar a coerência de
   fora), que são erros comuns e que a pipeline evita com razão medida.
2. **Os pontos fracos são conhecidos, não ocultos.** Cada falso positivo e falso negativo tem causa
   nomeável ("câmeras" ausente do léxico, "a paciente" de gênero comum, "os trabalho" absorvido pelo
   parser) — todas previstas como modos de falha das técnicas escolhidas. Um avaliador cujos erros
   você consegue explicar é um avaliador em que se pode confiar.
3. **A etapa semântica usa o vencedor do estudo.** Verbo mascarado com margem relativa: F1 0,91,
   decidindo sem escala absoluta — mais exato, mais robusto e 14× mais barato que a surpresa com
   limiar fixo.

**Ressalva de tamanho:** 36 frases de defeitos sintéticos medem *roteamento de defeito*, não
qualidade de texto real, e não substituem anotação humana. Os F1 aqui são indicativos, não
estimativas de produção. O valor do exercício é mostrar que **as escolhas da pipeline têm respaldo
empírico** e que o método — construir num conjunto, validar noutro, com gabarito — é o caminho para
sustentar qualquer avaliador.